# Modelo genérico · Recomendação baseada em conteúdo com rede neural (duas torres)

**Template reutilizável**: uma rede transforma as características do **usuário** num vetor $v_u$, outra transforma as do **item** num vetor $v_m$, e a nota prevista é o produto escalar dos dois (normalizados). Os mesmos vetores de item servem para encontrar **itens semelhantes**.

| | |
|---|---|
| **Modelo no sistema** | Machine Learning › 16. Sistemas de recomendação › cartão *Filtragem baseada em conteúdo* (`/modelar/sistemas-de-recomendacao/#m-content-based`) |
| **Origem** | `C3_W2_RecSysNN_Assignment.ipynb` (*Machine Learning Specialization*, DeepLearning.AI e Stanford Online; MovieLens reduzido: 397 usuários, 847 filmes, gêneros e ano), revisado, corrigido e generalizado |
| **Tipo de problema** | Recomendação · filtragem baseada em conteúdo (regressão da nota + ranking) |
| **Dependências** | `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `keras` 3 |

### Dados esperados
| tabela | colunas | observação |
|---|---|---|
| **avaliações** | usuário, item, nota | uma linha por avaliação |
| **itens** | item, colunas numéricas (ex.: ano, preço), coluna de categorias múltiplas (ex.: `"Ação|Comédia"`), título (opcional) | o "conteúdo" do item |

As características do **usuário** são construídas como no original: a nota média que ele dá a cada categoria. Com `CAMINHO_AVALIACOES = None`, o notebook gera dados sintéticos nesse formato.

### O que mudou em relação ao notebook original
- **Vazamento de dados corrigido:** os escalonadores (`StandardScaler` e `MinMaxScaler`) eram ajustados com **todas** as linhas antes da divisão treino/teste, e as médias por gênero e por filme também usavam todas as avaliações. Agora a divisão vem primeiro e tudo é calculado só com o treino.
- **Divisão frágil:** o original chamava `train_test_split` três vezes (itens, usuários, alvo) contando com a mesma semente para manter as linhas alinhadas. Agora há uma única divisão por índices.
- **Erro na tabela de itens semelhantes:** a coluna de gêneros do segundo filme repetia os gêneros do primeiro (`movie_dict[movie1_id]['genres']`). Corrigido.
- A matriz de distâncias usava dois laços Python (n² chamadas de `sq_dist`); virou uma operação vetorizada (`sq_dist` continua, conferida).
- Avaliação: além do MSE na escala normalizada, RMSE e MAE em unidades de nota, comparação com um *baseline* (média do item) e as métricas de ranking do cartão (precision@k, recall@k, NDCG@k) contra um baseline de popularidade.
- Keras 3: `Input(shape=(n,))` com tupla (o original passava um inteiro) e `UnitNormalization` no lugar de `tf.linalg.l2_normalize` dentro do modelo funcional. `recsysNN_utils.py`, `public_tests.py` e os CSVs do curso não são mais necessários.

## 1. Configuração

In [ ]:
# >>> ADAPTAR: única célula que precisa ser editada para usar outro dataset.

# --- avaliações ---
CAMINHO_AVALIACOES = None    # ex.: "dados/avaliacoes.csv"; None = dados sintéticos
COLUNA_USUARIO = "usuario_id"
COLUNA_ITEM = "item_id"
COLUNA_NOTA = "nota"

# --- itens (conteúdo) ---
CAMINHO_ITENS = None         # ex.: "dados/itens.csv" (obrigatório se CAMINHO_AVALIACOES não for None)
COLUNA_ITEM_NA_TABELA = "item_id"
COLUNA_CATEGORIAS = "categorias"   # coluna com várias categorias por item; None se não houver
SEPARADOR_CATEGORIAS = "|"
COLUNAS_ITEM_NUMERICAS = None      # None = todas as numéricas da tabela de itens (exceto o id)
COLUNA_TITULO = "titulo"           # para exibição; None se não houver
SEPARADOR = None                   # separador dos arquivos CSV; None = detecta

# --- modelo ---
CAMADAS = [256, 128]         # camadas ocultas de cada torre (o original usava 256 e 128)
NUM_SAIDAS = 32              # dimensão dos vetores v_u e v_m
TAXA_APRENDIZADO = 0.01
EPOCAS = 30
TAMANHO_LOTE = 128
TAMANHO_TESTE = 0.2
SEMENTE = 1

# --- uso ---
K = 10                       # tamanho das listas de recomendação
LIMIAR_RELEVANTE = 4.0       # nota a partir da qual um item conta como "relevante" nas métricas de ranking
MAX_USUARIOS_AVALIACAO = 300 # usuários amostrados para as métricas de ranking
NOVO_USUARIO = None          # ex.: {"Aventura": 5.0, "Fantasia": 5.0}; None = usa as duas primeiras categorias

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import pandas as pd
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

keras.utils.set_random_seed(SEMENTE)
rng = np.random.default_rng(SEMENTE)

## 2. Dados

In [ ]:
def carregar_tabela(caminho, separador=None, tem_cabecalho=True, nomes_colunas=None, planilha=0):
    """Lê CSV/TXT/TSV, Excel, Parquet ou JSON e devolve um DataFrame."""
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho.resolve()}")
    ext = caminho.suffix.lower()
    cabecalho = 0 if tem_cabecalho else None
    if ext in {".csv", ".txt", ".tsv"}:
        sep = separador if separador is not None else ("\t" if ext == ".tsv" else None)
        df = pd.read_csv(caminho, sep=sep, engine="python", header=cabecalho)
    elif ext in {".xlsx", ".xls"}:
        df = pd.read_excel(caminho, sheet_name=planilha, header=cabecalho)
    elif ext == ".parquet":
        df = pd.read_parquet(caminho)
    elif ext == ".json":
        df = pd.read_json(caminho)
    else:
        raise ValueError(f"Formato não suportado: {ext}")
    if nomes_colunas is not None:
        df.columns = nomes_colunas
    elif not tem_cabecalho:
        df.columns = [f"coluna_{i}" for i in range(df.shape[1])]
    return df


def dados_sinteticos(n_usuarios=300, n_itens=400, por_usuario=40, semente=SEMENTE):
    """Itens com 1 a 3 categorias e um ano; usuários com gosto latente por categoria; notas de 0,5 a 5."""
    r = np.random.default_rng(semente)
    cats = ["Ação", "Aventura", "Animação", "Comédia", "Drama", "Fantasia", "Suspense", "Romance"]
    multi = np.zeros((n_itens, len(cats)), int)
    for i in range(n_itens):
        multi[i, r.choice(len(cats), size=r.integers(1, 4), replace=False)] = 1
    itens = pd.DataFrame({"item_id": np.arange(n_itens), "titulo": [f"Item {i}" for i in range(n_itens)],
                          "ano": r.integers(2000, 2021, n_itens), "duracao": r.normal(110, 15, n_itens).round(),
                          "categorias": ["|".join(c for c, f in zip(cats, linha) if f) for linha in multi]})
    qualidade = r.normal(0, 0.4, n_itens)
    gosto = r.normal(0, 1.0, (n_usuarios, len(cats)))
    linhas = []
    for u in range(n_usuarios):
        vistos = r.choice(n_itens, size=por_usuario, replace=False)
        afinidade = (multi[vistos] * gosto[u]).sum(1) / multi[vistos].sum(1)
        nota = 3.3 + 0.9 * afinidade + qualidade[vistos] + r.normal(0, 0.45, por_usuario)
        for i, n in zip(vistos, np.clip(np.round(nota * 2) / 2, 0.5, 5.0)):
            linhas.append((u, i, n))
    return pd.DataFrame(linhas, columns=["usuario_id", "item_id", "nota"]), itens


if CAMINHO_AVALIACOES is None:
    avaliacoes, itens = dados_sinteticos()
else:
    if CAMINHO_ITENS is None:
        raise ValueError("Informe CAMINHO_ITENS: a filtragem por conteúdo precisa das características dos itens.")
    avaliacoes = carregar_tabela(CAMINHO_AVALIACOES, SEPARADOR)
    itens = carregar_tabela(CAMINHO_ITENS, SEPARADOR)

avaliacoes = avaliacoes.rename(columns={COLUNA_USUARIO: "usuario", COLUNA_ITEM: "item", COLUNA_NOTA: "nota"})
itens = itens.rename(columns={COLUNA_ITEM_NA_TABELA: "item"}).drop_duplicates("item").set_index("item")
avaliacoes = avaliacoes.dropna(subset=["usuario", "item", "nota"])
fora = ~avaliacoes["item"].isin(itens.index)
if fora.any():
    print(f"{fora.sum()} avaliações de itens sem conteúdo descartadas")
avaliacoes = avaliacoes[~fora].reset_index(drop=True)
print(f"{avaliacoes.usuario.nunique()} usuários | {len(itens)} itens | {len(avaliacoes)} avaliações | "
      f"notas de {avaliacoes.nota.min()} a {avaliacoes.nota.max()}")

In [ ]:
# Conteúdo dos itens: colunas numéricas + categorias em multi-hot (um indicador por categoria).
if COLUNA_CATEGORIAS is not None:
    multi_hot = itens[COLUNA_CATEGORIAS].fillna("").str.get_dummies(sep=SEPARADOR_CATEGORIAS)
    multi_hot = multi_hot.loc[:, [c for c in multi_hot.columns if c and not c.startswith("(")]]
else:
    multi_hot = pd.DataFrame(index=itens.index)
categorias = list(multi_hot.columns)
numericas_item = COLUNAS_ITEM_NUMERICAS or [c for c in itens.select_dtypes(include="number").columns]
print("categorias:", categorias)
print("numéricas do item:", numericas_item)

# Visões gerais, como as tabelas do original: mais avaliados e resumo por categoria.
por_item = avaliacoes.groupby("item").nota.agg(["count", "mean"]).rename(columns={"count": "avaliações", "mean": "nota média"})
if COLUNA_TITULO:
    por_item.insert(0, "título", itens.loc[por_item.index, COLUNA_TITULO])
display(por_item.sort_values("avaliações", ascending=False).head(10).round(2))
if categorias:
    expl = avaliacoes.join(multi_hot, on="item")
    display(pd.DataFrame({"itens": multi_hot.sum(), "avaliações": expl[categorias].sum(),
                          "nota média": [expl.loc[expl[c] == 1, "nota"].mean() for c in categorias]}).round(2))

## 3. Divisão e características (só com o treino)
Uma única divisão das avaliações. Depois, com **apenas o treino**: nota média de cada item (característica derivada do item, como no original) e nota média de cada usuário por categoria (características do usuário). Itens ou usuários sem avaliação no treino recebem a média geral ou zero, como um caso de partida a frio.

In [ ]:
i_tr, i_te = train_test_split(np.arange(len(avaliacoes)), test_size=TAMANHO_TESTE, random_state=SEMENTE)
treino, teste = avaliacoes.iloc[i_tr], avaliacoes.iloc[i_te]
media_global = treino.nota.mean()

# Itens: numéricas + média do item no treino + categorias
media_item = treino.groupby("item").nota.mean()
item_feats = pd.concat([itens[numericas_item].astype(float),
                        media_item.reindex(itens.index).fillna(media_global).rename("nota_media_item"),
                        multi_hot.astype(float)], axis=1)
item_feats = item_feats.fillna(item_feats.median())

# Usuários: nota média por categoria no treino (0 = categoria não avaliada, como no original)
if categorias:
    expl_tr = treino.join(multi_hot, on="item")
    soma = expl_tr[categorias].mul(expl_tr.nota, axis=0).groupby(expl_tr.usuario).sum()
    cont = expl_tr[categorias].groupby(expl_tr.usuario).sum()
    user_feats = (soma / cont.replace(0, np.nan)).fillna(0.0)
else:
    user_feats = treino.groupby("usuario").nota.mean().to_frame("nota_media")
user_feats.columns = [f"media_{c}" for c in user_feats.columns]
# contagem e média geral ficam só para interpretação (não entram no treino, como no original)
resumo_usuario = treino.groupby("usuario").nota.agg(["count", "mean"])


def vetores(linhas):
    U = user_feats.reindex(linhas.usuario).fillna(0.0).to_numpy()
    I = item_feats.loc[linhas.item].to_numpy()
    return U, I, linhas.nota.to_numpy(dtype=float)


U_tr, I_tr, y_tr = vetores(treino)
U_te, I_te, y_te = vetores(teste)
print(f"treino: {len(y_tr)} | teste: {len(y_te)} | features de usuário: {U_tr.shape[1]} | de item: {I_tr.shape[1]}")

In [ ]:
# Escalonamento ajustado SÓ no treino. Alvo levado para [-1, 1], como no original.
scalerUser, scalerItem = StandardScaler().fit(U_tr), StandardScaler().fit(I_tr)
scalerTarget = MinMaxScaler((-1, 1)).fit(y_tr.reshape(-1, 1))
U_tr_s, U_te_s = scalerUser.transform(U_tr), scalerUser.transform(U_te)
I_tr_s, I_te_s = scalerItem.transform(I_tr), scalerItem.transform(I_te)
y_tr_s = scalerTarget.transform(y_tr.reshape(-1, 1))
y_te_s = scalerTarget.transform(y_te.reshape(-1, 1))
assert np.allclose(scalerItem.inverse_transform(I_tr_s), I_tr)     # a transformação é reversível

## 4. Rede de duas torres
Cada torre é uma rede densa (ReLU nas ocultas, saída linear com `NUM_SAIDAS` unidades). Os vetores são normalizados para norma 1, então o produto escalar é a similaridade de cosseno entre usuário e item. As torres não precisam ser iguais: com muito mais conteúdo de um lado, aumente a torre correspondente.

In [ ]:
def torre(nome):
    return keras.Sequential([layers.Dense(u, activation="relu") for u in CAMADAS] + [layers.Dense(NUM_SAIDAS)], name=nome)


user_NN, item_NN = torre("torre_usuario"), torre("torre_item")
input_user = keras.Input(shape=(U_tr_s.shape[1],), name="usuario")
input_item = keras.Input(shape=(I_tr_s.shape[1],), name="item")
vu = layers.UnitNormalization(axis=1)(user_NN(input_user))
vm = layers.UnitNormalization(axis=1)(item_NN(input_item))
output = layers.Dot(axes=1)([vu, vm])
model = keras.Model([input_user, input_item], output)
model.summary()

model.compile(optimizer=keras.optimizers.Adam(learning_rate=TAXA_APRENDIZADO), loss=keras.losses.MeanSquaredError())
hist = model.fit([U_tr_s, I_tr_s], y_tr_s, validation_split=0.1, epochs=EPOCAS, batch_size=TAMANHO_LOTE, verbose=0)
plt.figure(figsize=(6, 3.5))
plt.plot(hist.history["loss"], label="treino"); plt.plot(hist.history["val_loss"], label="validação")
plt.xlabel("época"); plt.ylabel("MSE (escala [-1, 1])"); plt.legend(); plt.title("Curvas de perda"); plt.show()

## 5. Avaliação
**Erro da nota:** MSE na escala do treino (como o original) e RMSE/MAE em unidades de nota, ao lado de um *baseline* que prevê a nota média do item. **Ranking** (métricas do cartão): para cada usuário, os itens que ele não avaliou no treino são ordenados pela nota prevista; acerta quem aparece entre os *K* primeiros e foi avaliado no teste com nota ≥ `LIMIAR_RELEVANTE`.

In [ ]:
mse_escalado = model.evaluate([U_te_s, I_te_s], y_te_s, verbose=0)
pred_te = scalerTarget.inverse_transform(model.predict([U_te_s, I_te_s], verbose=0)).ravel()
base_te = media_item.reindex(teste.item).fillna(media_global).to_numpy()
rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
display(pd.DataFrame({"rede (duas torres)": {"RMSE": rmse(y_te, pred_te), "MAE": float(np.mean(np.abs(y_te - pred_te)))},
                      "baseline: média do item": {"RMSE": rmse(y_te, base_te), "MAE": float(np.mean(np.abs(y_te - base_te)))}}).T.round(4))
print(f"MSE na escala [-1, 1] (comparável ao original): {mse_escalado:.4f}")

In [ ]:
I_all_s = scalerItem.transform(item_feats.to_numpy())
popularidade = treino.groupby("item").size().reindex(item_feats.index).fillna(0).to_numpy()


def metricas_ranking(scores, vistos, relevantes, k=K):
    s = scores.copy()
    s[vistos] = -np.inf                                   # não recomenda o que já foi consumido no treino
    topo = np.argsort(-s)[:k]
    acertos = np.isin(topo, relevantes)
    dcg = (acertos / np.log2(np.arange(2, k + 2))).sum()
    idcg = (1 / np.log2(np.arange(2, min(len(relevantes), k) + 2))).sum()
    return acertos.sum() / k, acertos.sum() / len(relevantes), dcg / idcg


pos_item = {it: j for j, it in enumerate(item_feats.index)}
rel_te = teste[teste.nota >= LIMIAR_RELEVANTE].groupby("usuario").item.apply(lambda s: [pos_item[i] for i in s])
vistos_tr = treino.groupby("usuario").item.apply(lambda s: [pos_item[i] for i in s])
usuarios_aval = [u for u in rel_te.index if u in vistos_tr.index]
usuarios_aval = list(rng.choice(usuarios_aval, size=min(MAX_USUARIOS_AVALIACAO, len(usuarios_aval)), replace=False))
res = {"rede": [], "popularidade": []}
for u in usuarios_aval:
    U_u = scalerUser.transform(np.repeat(user_feats.loc[[u]].to_numpy(), len(I_all_s), axis=0))
    s_rede = model.predict([U_u, I_all_s], verbose=0, batch_size=4096).ravel()
    res["rede"].append(metricas_ranking(s_rede, vistos_tr[u], rel_te[u]))
    res["popularidade"].append(metricas_ranking(popularidade.astype(float), vistos_tr[u], rel_te[u]))
display(pd.DataFrame({k: dict(zip([f"precision@{K}", f"recall@{K}", f"NDCG@{K}"], np.mean(v, axis=0)))
                      for k, v in res.items()}).T.round(4))
print(f"{len(usuarios_aval)} usuários avaliados")

## 6. Previsões
### 6.1 Novo usuário
Um usuário novo é descrito pela nota média que daria a cada categoria (0 = sem opinião). Altere `NOVO_USUARIO` com as suas preferências.

In [ ]:
prefs = NOVO_USUARIO or ({categorias[0]: 5.0, categorias[1]: 5.0} if len(categorias) >= 2 else {})
user_vec = pd.DataFrame(0.0, index=[0], columns=user_feats.columns)
for cat, nota in prefs.items():
    if f"media_{cat}" not in user_vec.columns:
        raise KeyError(f"categoria {cat!r} não existe; disponíveis: {categorias}")
    user_vec[f"media_{cat}"] = nota

user_vecs = np.repeat(user_vec.to_numpy(), len(item_feats), axis=0)        # replica para todos os itens
y_p = model.predict([scalerUser.transform(user_vecs), I_all_s], verbose=0)
y_pu = scalerTarget.inverse_transform(y_p).ravel()
ordem = np.argsort(-y_pu)[:K]
print("preferências:", prefs)
tabela = item_feats.iloc[ordem][["nota_media_item"]].assign(nota_prevista=y_pu[ordem].round(2))
if COLUNA_TITULO:
    tabela.insert(0, "título", itens.loc[tabela.index, COLUNA_TITULO])
if COLUNA_CATEGORIAS:
    tabela["categorias"] = itens.loc[tabela.index, COLUNA_CATEGORIAS]
display(tabela)

### 6.2 Usuário existente: previsto × real

In [ ]:
uid = teste.usuario.iloc[0]                       # >>> ADAPTAR: id de um usuário do seu dataset
linhas_u = avaliacoes[avaliacoes.usuario == uid]
U_u, I_u, y_u = vetores(linhas_u)
p_u = scalerTarget.inverse_transform(model.predict([scalerUser.transform(U_u), scalerItem.transform(I_u)], verbose=0)).ravel()
comp = linhas_u.assign(prevista=p_u.round(2), conjunto=np.where(linhas_u.index.isin(teste.index), "teste", "treino"))
if COLUNA_TITULO:
    comp["título"] = itens.loc[comp.item, COLUNA_TITULO].to_numpy()
display(comp.sort_values("prevista", ascending=False).head(15))
print(f"usuário {uid}: {int(resumo_usuario.loc[uid, 'count']) if uid in resumo_usuario.index else 0} avaliações no treino")

### 6.3 Itens semelhantes
Itens parecidos têm vetores $v_m$ próximos. A distância ao quadrado entre dois vetores é

$$\lVert v_m^{(k)} - v_m^{(i)}\rVert^2 = \sum_{l=1}^{n}\left(v_{m_l}^{(k)} - v_{m_l}^{(i)}\right)^2$$

A matriz de distâncias pode ser calculada uma vez, depois do treino, e reutilizada sem retreinar.

In [ ]:
def sq_dist(a, b):
    """Distância euclidiana ao quadrado entre dois vetores."""
    return float(np.sum(np.square(a - b)))


assert sq_dist(np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 3.0])) == 0
assert np.isclose(sq_dist(np.array([1.1, 2.1, 3.1]), np.array([1.0, 2.0, 3.0])), 0.03)
assert sq_dist(np.array([0, 1, 0]), np.array([1, 0, 0])) == 2

input_item_m = keras.Input(shape=(I_all_s.shape[1],))
model_m = keras.Model(input_item_m, layers.UnitNormalization(axis=1)(item_NN(input_item_m)))   # usa a torre treinada
vms = model_m.predict(I_all_s, verbose=0)
print("vetores de item:", vms.shape)

sq = (vms ** 2).sum(axis=1)
dist = np.maximum(sq[:, None] + sq[None, :] - 2 * vms @ vms.T, 0)        # todas as distâncias de uma vez
assert np.isclose(dist[0, 1], sq_dist(vms[0], vms[1]), atol=1e-5)
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))         # ignora a distância do item a ele mesmo

nome = (lambda j: itens.loc[item_feats.index[j], COLUNA_TITULO]) if COLUNA_TITULO else (lambda j: item_feats.index[j])
cats_de = (lambda j: itens.loc[item_feats.index[j], COLUNA_CATEGORIAS]) if COLUNA_CATEGORIAS else (lambda j: "")
linhas = []
for i in range(min(15, len(vms))):
    j = int(np.argmin(m_dist[i]))
    linhas.append({"item 1": nome(i), "categorias 1": cats_de(i), "item 2 (mais próximo)": nome(j),
                   "categorias 2": cats_de(j)})                           # correção: categorias do item 2
display(pd.DataFrame(linhas))

## 7. Próximos passos (segundo o cartão do sistema)
- O conteúdo do usuário pode ir muito além das médias por categoria (dados cadastrais, comportamento recente); o do item, incluir texto (TF-IDF, *embeddings*) e imagens.
- Na produção: calcule os vetores dos itens com antecedência e faça a recuperação por vizinhos mais próximos; depois, reordene os candidatos com o modelo completo.
- Acompanhe **diversidade** e **novidade** das listas: o conteúdo tende à superespecialização.
- **Prefira filtragem colaborativa** quando houver histórico abundante e conteúdo pobre; **híbrido** na maioria dos casos de produção.